In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Thiết lập seed để kết quả có thể tái tạo
torch.manual_seed(42)

In [2]:
# Hyperparameters cơ bản
N = 4      # Độ dài chuỗi (Sequence length)
d = 512    # Số chiều của Model (Model dimensionality)

# Khởi tạo ma trận đầu vào X (thường là tổng của Token Embedding + Positional Embedding)
# Ở đây ta tạo random để mô phỏng
X = torch.rand(N, d)
print(f"Shape của đầu vào X: {X.shape}")

Shape của đầu vào X: torch.Size([4, 512])


In [3]:
# Giả sử ta dùng 1 head, số chiều d_k và d_v thường nhỏ hơn d (ví dụ: 64)
d_k = 64
d_v = 64

# Các Linear Layer đại diện cho các ma trận trọng số W^Q, W^K, W^V
W_Q = nn.Linear(d, d_k, bias=False)
W_K = nn.Linear(d, d_k, bias=False)
W_V = nn.Linear(d, d_v, bias=False)

# Tính toán ma trận Q, K, V
Q = W_Q(X)  # Shape: [N x d_k]
K = W_K(X)  # Shape: [N x d_k]
V = W_V(X)  # Shape: [N x d_v]

print(f"Shape của Q: {Q.shape}")

Shape của Q: torch.Size([4, 64])


In [4]:
# 1. Tính toán điểm số thô (Raw scores)
scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
print(f"Shape của điểm số QK^T: {scores.shape} (N x N)\n")

# 2. Tạo Causal Mask (ma trận tam giác trên chứa giá trị -inf)
mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
masked_scores = scores.masked_fill(mask, float('-inf'))
print("Điểm số sau khi áp dụng Mask (che tương lai):")
print(masked_scores)

# 3. Tính phân phối Softmax (Attention Weights)
attention_weights = F.softmax(masked_scores, dim=-1)
print("\nTrọng số Attention sau Softmax (tổng mỗi hàng = 1):")
print(torch.round(attention_weights * 100) / 100) # Làm tròn cho dễ nhìn

Shape của điểm số QK^T: torch.Size([4, 4]) (N x N)

Điểm số sau khi áp dụng Mask (che tương lai):
tensor([[0.0599,   -inf,   -inf,   -inf],
        [0.0989, 0.1536,   -inf,   -inf],
        [0.0271, 0.0782, 0.0708,   -inf],
        [0.0849, 0.1325, 0.1181, 0.0754]], grad_fn=<MaskedFillBackward0>)

Trọng số Attention sau Softmax (tổng mỗi hàng = 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4900, 0.5100, 0.0000, 0.0000],
        [0.3200, 0.3400, 0.3400, 0.0000],
        [0.2500, 0.2600, 0.2500, 0.2400]], grad_fn=<DivBackward0>)


In [5]:
# 4. Tính toán đầu ra của Attention Head
A = torch.matmul(attention_weights, V)

print(f"\nShape của đầu ra Attention A: {A.shape} (N x d_v)")
# Lưu ý: Trong Multi-Head Attention thực tế, ta sẽ concat nhiều head A lại
# và nhân với ma trận W^O để đưa về lại shape [N x d].


Shape của đầu ra Attention A: torch.Size([4, 64]) (N x d_v)
